In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split


np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'ecg_hr': np.random.normal(75, 15, n_samples),
    'bp_sys': np.random.normal(120, 15, n_samples),
    'bp_dia': np.random.normal(80, 10, n_samples),
    'spo2': np.clip(np.random.normal(97, 3, n_samples), 70, 100),
    'temperature': np.random.normal(37.0, 0.8, n_samples),
    # Mocking RGB values (healthy is around [255, 234, 112])
    'urine_r': np.random.normal(255, 5, n_samples),
    'urine_g': np.random.normal(230, 20, n_samples),
    'urine_b': np.random.normal(110, 30, n_samples)
})


def assign_triage(row):
    # RED
    if row['spo2'] < 90 or row['ecg_hr'] > 130 or row['temperature'] > 39.5 or row['bp_sys'] > 180:
        return 2
    # YELLOW
    elif (row['spo2'] < 95) or (row['temperature'] > 38.0) or (row['bp_sys'] > 140):
        return 1
    # GREEN
    else:
        return 0

data['triage_class'] = data.apply(assign_triage, axis=1)

X = data.drop('triage_class', axis=1)
y = data['triage_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=3,
    learning_rate=0.1,
    n_estimators=50
)

model.fit(X_train, y_train)


model.save_model('triage_xgboost.json')
print("Model trained and saved as 'triage_xgboost.json'")

Model trained and saved as 'triage_xgboost.json'


We will flip 10% of the labes that we are going to use for training so that the model is not 100% sure of the things which might lead to hallucination we are also going to reduce the learning rate for a smoother result.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples = 2000 

data = pd.DataFrame({
    'ecg_hr': np.random.normal(85, 25, n_samples),
    'bp_sys': np.random.normal(125, 20, n_samples),
    'bp_dia': np.random.normal(80, 15, n_samples),
    'spo2': np.clip(np.random.normal(95, 5, n_samples), 70, 100),
    'temperature': np.random.normal(37.5, 1.2, n_samples),
    'urine_r': np.random.normal(255, 5, n_samples),
    'urine_g': np.random.normal(230, 20, n_samples),
    'urine_b': np.random.normal(110, 30, n_samples)
})

def assign_triage(row):
    # RED 
    if row['spo2'] <= 90 or row['ecg_hr'] >= 130 or row['temperature'] >= 39.0 or row['bp_sys'] >= 160:
        return 2
    # YELLOW 
    elif (row['spo2'] <= 96) or (row['ecg_hr'] >= 100) or (row['temperature'] >= 37.8) or (row['bp_sys'] >= 135):
        return 1
    # GREEN
    else:
        return 0

data['triage_class'] = data.apply(assign_triage, axis=1)


noise_indices = np.random.choice(data.index, size=int(n_samples * 0.10), replace=False)
data.loc[noise_indices, 'triage_class'] = np.random.choice([0, 1, 2], size=len(noise_indices))

X = data.drop('triage_class', axis=1)
y = data['triage_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=4, 
    learning_rate=0.05, 
    n_estimators=100,
    subsample=0.8,      # Using 80% of data per tree adds variance
    colsample_bytree=0.8
)

model.fit(X_train, y_train)


model.save_model('triage_xgboost.json')
print("Upgraded Model saved as 'triage_xgboost.json'")

Upgraded 'Fuzzy' Model trained and saved as 'triage_xgboost.json'


Refined range by creating overlapping datasets as having sharp boundaries makes the model get high confidence rates, doing this by inc in SD, making changes such as shuffeling the data inorder for the AI not to recognise patterns from previous sets also reducing the depth and no of trees for less memorization and underfitting the data.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

np.random.seed(42)

green = pd.DataFrame({
    'ecg_hr': np.random.normal(75, 12, 1000),
    'bp_sys': np.random.normal(115, 10, 1000),
    'bp_dia': np.random.normal(75, 8, 1000),
    'spo2': np.clip(np.random.normal(98, 1.5, 1000), 90, 100),
    'temperature': np.random.normal(37.0, 0.4, 1000),
    'urine_r': 255.0, 'urine_g': 234.0, 'urine_b': 112.0,
    'triage_class': 0
})

yellow = pd.DataFrame({
    'ecg_hr': np.random.normal(95, 15, 800),
    'bp_sys': np.random.normal(135, 15, 800),
    'bp_dia': np.random.normal(85, 10, 800),
    'spo2': np.clip(np.random.normal(94, 2.5, 800), 85, 100),
    'temperature': np.random.normal(37.8, 0.6, 800),
    'urine_r': 255.0, 'urine_g': 234.0, 'urine_b': 112.0,
    'triage_class': 1
})


red = pd.DataFrame({
    'ecg_hr': np.random.normal(135, 20, 600),
    'bp_sys': np.random.normal(155, 20, 600),
    'bp_dia': np.random.normal(95, 15, 600),
    'spo2': np.clip(np.random.normal(88, 4, 600), 70, 100),
    'temperature': np.random.normal(39.0, 0.8, 600),
    'urine_r': 255.0, 'urine_g': 234.0, 'urine_b': 112.0,
    'triage_class': 2
})


data = pd.concat([green, yellow, red], ignore_index=True)


data = data.sample(frac=1, random_state=42).reset_index(drop=True)

X = data.drop('triage_class', axis=1)
y = data['triage_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=2,          
    learning_rate=0.05,   
    n_estimators=30,      
    subsample=0.7,        
    colsample_bytree=0.7
)

model.fit(X_train, y_train)
model.save_model('triage_xgboost.json')
print("Overlapping Model trained and saved as 'triage_xgboost.json'")

Overlapping Model trained and saved as 'triage_xgboost.json'
